# All-Methods Comparison — BTC 4-Hourly (Price + Return mode)

Runs **all 13 model variants** across **both target modes** and **3 seeds**, then aggregates results.

**Model families (13 variants):**

| Family | Variants | Data config | Features |
|---|---|---|---|
| OHLC comparison | naive_zero, real_lstm (±attn), quaternion_lstm (±attn), 2 param-matched = **7** | `btc_ohlc` | 4 (OHLC) |
| Hierarchical | concat, concat_attn, group_attn, group_attn_temporal, meta_quat, meta_quat_attn = **6** | `btc_hier` | 16 (LunarCrush) |

**4 experiments total** (2 model families × 2 target modes), each with **3 seeds** → mean ± std + significance vs `real_lstm`.

| # | Experiment | Base config | Target |
|---|---|---|---|
| 1 | OHLC · price | `btc_ohlc.yaml` | next close |
| 2 | OHLC · return | `btc_ohlc_return.yaml` | return |
| 3 | Hierarchical · price | `btc_hier.yaml` | next close |
| 4 | Hierarchical · return | `btc_hier_return.yaml` | return |

**Runtime:** heavy. 13 variants × 3 seeds × 4 experiments on a T4 is a few hours. Keep the tab alive.

**Requires:** `lunarcrush_btc_4hour_full.csv` uploaded to Google Drive (see cell 1).

## 1. Setup

In [ ]:
import os
if not os.path.exists('/content/thesis'):
    !git clone https://github.com/BerkayClik/thesis.git /content/thesis
%cd /content/thesis
!git pull

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
# Copy the BTC 4-hour cache CSV from Drive (recursive search).
import os, shutil, glob
os.makedirs('data/cache', exist_ok=True)
DRIVE_DATA_DIR = '/content/drive/MyDrive/thesis_data'   # <-- the folder you uploaded to
copied = 0
if os.path.isdir(DRIVE_DATA_DIR):
    for src in glob.glob(os.path.join(DRIVE_DATA_DIR, '**/lunarcrush_btc_4hour*.csv'), recursive=True):
        shutil.copy(src, os.path.join('data/cache', os.path.basename(src))); copied += 1
        print('copied', os.path.basename(src))
if copied == 0:
    print(f'No BTC 4-hour CSV found under {DRIVE_DATA_DIR} (searched recursively).')
!ls -la data/cache/lunarcrush_btc_4hour*.csv 2>/dev/null || echo 'missing BTC 4h cache'

In [ ]:
!pip install -q yfinance scipy seaborn
import torch
print('PyTorch:', torch.__version__, '| CUDA:', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
os.environ['PYTHONPATH'] = '/content/thesis'
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'

## 2. Run all 4 experiments (3 seeds each)

Each cell runs one experiment. The 3-seed configs and the return-mode data configs are already committed in the repo. Results land under `experiments/results/`.

In [ ]:
# 1/4 — OHLC · price-mode (7 variants × 3 seeds)
!python experiments/run_experiments.py \
    --base-config configs/data/4hourly/btc_ohlc.yaml \
    --experiment-config configs/experiments/4hourly_comparison_3seed.yaml

In [ ]:
# 2/4 — OHLC · return-mode (7 variants × 3 seeds)
!python experiments/run_experiments.py \
    --base-config configs/data/4hourly/btc_ohlc_return.yaml \
    --experiment-config configs/experiments/4hourly_comparison_3seed.yaml

In [ ]:
# 3/4 — Hierarchical · price-mode (6 variants × 3 seeds)
!python experiments/run_experiments.py \
    --base-config configs/data/4hourly/btc_hier.yaml \
    --experiment-config configs/experiments/4hourly_hierarchical_3seed.yaml

In [ ]:
# 4/4 — Hierarchical · return-mode (6 variants × 3 seeds)
!python experiments/run_experiments.py \
    --base-config configs/data/4hourly/btc_hier_return.yaml \
    --experiment-config configs/experiments/4hourly_hierarchical_3seed.yaml

## 3. Aggregate results into one comparison table

Pulls the latest JSON from each of the 4 result dirs and builds a single dataframe of mean ± std per variant per experiment.

In [ ]:
import json, glob, os
import numpy as np, pandas as pd

EXPERIMENTS = {
    'OHLC·price':  'experiments/results/4hourly_btc_ohlc',
    'OHLC·return': 'experiments/results/4hourly_btc_ohlc_return',
    'Hier·price':  'experiments/results/4hourly_btc_hier',
    'Hier·return': 'experiments/results/4hourly_btc_hier_return',
}
METRICS = ['mape', 'directional_accuracy', 'sharpe_ratio',
           'directional_accuracy_3class', 'sharpe_ratio_3class']

rows = []
for exp_name, d in EXPERIMENTS.items():
    js = [f for f in glob.glob(f'{d}/*.json') if 'intermediate' not in f]
    if not js:
        print(f'[skip] no results in {d}')
        continue
    res = json.load(open(max(js, key=os.path.getmtime)))
    for variant, vdata in res['model_results'].items():
        agg = vdata.get('aggregated', {})
        row = {'experiment': exp_name, 'variant': variant}
        for m in METRICS:
            md = agg.get(m, {})
            row[m] = md.get('mean', np.nan)
            row[m + '_std'] = md.get('std', np.nan)
        rows.append(row)

df = pd.DataFrame(rows)
pd.set_option('display.width', 200, 'display.max_columns', 30)
df_disp = df[['experiment', 'variant', 'mape', 'directional_accuracy',
              'sharpe_ratio', 'directional_accuracy_3class', 'sharpe_ratio_3class']].round(3)
df_disp

In [ ]:
# Signal quality (corr + directional agreement) per variant, from saved predictions
sig_rows = []
for exp_name, d in EXPERIMENTS.items():
    js = [f for f in glob.glob(f'{d}/*.json') if 'intermediate' not in f]
    if not js:
        continue
    res = json.load(open(max(js, key=os.path.getmtime)))
    for variant, vdata in res['model_results'].items():
        tm = vdata['individual_runs'][0]['test_metrics']
        if not tm.get('predictions'):
            continue
        preds = np.array(tm['predictions'], float)
        targs = np.array(tm['targets'], float)
        prevs = np.array(tm['prev_closes'], float)
        pr = preds / prevs - 1
        tr = targs / prevs - 1
        if pr.std() == 0 or tr.std() == 0:
            corr = np.nan
        else:
            corr = np.corrcoef(pr, tr)[0, 1]
        da = (np.sign(pr) == np.sign(tr)).mean() * 100
        sig_rows.append({'experiment': exp_name, 'variant': variant,
                         'corr': round(corr, 4), 'dir_agree_%': round(da, 1),
                         'pred_std/true_std': round(pr.std() / tr.std(), 3) if tr.std() else np.nan})
pd.DataFrame(sig_rows)

## 4. Comparison figures (per experiment)

Uses the repo's `visualize_results.py`. Note: it needs ≥ 2 variants to render the multi-model comparison plots, which all 4 experiments satisfy (6–7 variants each).

In [ ]:
from IPython.display import Image, display

for exp_name, d in EXPERIMENTS.items():
    js = [f for f in glob.glob(f'{d}/*.json') if 'intermediate' not in f]
    if not js:
        continue
    latest = max(js, key=os.path.getmtime)
    out = f'{d}/figures'
    print(f'\n=== {exp_name} ===')
    !python experiments/visualize_results.py --results "{latest}" --output "{out}" 2>&1 | tail -2
    for fig in ['metric_comparison.png', 'box_plots.png', 'radar_chart.png']:
        p = f'{out}/{fig}'
        if os.path.exists(p):
            display(Image(filename=p, width=850))

## 5. Save everything to Google Drive

In [ ]:
import shutil
from datetime import datetime

GDRIVE = '/content/drive/MyDrive/thesis_results_all_methods_4h'
run_dir = f"{GDRIVE}/{datetime.now().strftime('%Y%m%d_%H%M%S')}"
os.makedirs(run_dir, exist_ok=True)
for exp_name, d in EXPERIMENTS.items():
    if os.path.exists(d):
        shutil.copytree(d, f"{run_dir}/{os.path.basename(d)}", dirs_exist_ok=True)
        print('saved', exp_name)
# also save the two aggregate tables
df.to_csv(f'{run_dir}/comparison_table.csv', index=False)
print(f'\nAll saved to: {run_dir}')

## Notes

- **13 variants × 4 experiments × 3 seeds.** OHLC experiments use 4-feature quaternion encoding; hierarchical use the 16-feature LunarCrush grouping.
- **Price vs return** are directly comparable: in return-mode the predicted return is reconstructed to a price (`prev_close * (1 + r)`) before metrics, so MAPE / Sharpe are on the same scale.
- The legacy `test_metrics.sharpe_ratio` is the toy sign-based Sharpe (kept for comparability). For a fee-aware portfolio backtest, use `notebooks/Returns_Backtest_BTC_4Hourly.ipynb`.
- To also backtest these models, point the backtest adapter at any `*_predictions.csv` produced above.